In [10]:
import sys
sys.path.append("..")

In [11]:
import tqdm
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [ ]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, x_r, theta_0):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

In [13]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)
    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, i, x_0, x_r, theta_0)

    df_results = pd.DataFrame(results)
    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f'../results/recourse/lr_{dataset.name}_{recourse.name}_{seed}.pkl')
    
    return df_results

In [14]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
            results.append(df_results)

In [ ]:
torch.manual_seed(0)

d_results = {}
params = {}
params['alpha'] = 0.5 # float, None
params['lamb'] = 0.1
params['seeds'] = range(1)
params['save_results'] = False

datasets = [SyntheticDataset()]
recourse_fns = [LARRecourse, ROAR, ]

for dataset in datasets:
    results = []
    print(f'Running {dataset.name} data...')
    run_experiment(dataset, recourse_fns, params, results)
    
    d_results[dataset.name] = pd.concat(results)
    print(f'Finished {dataset.name}\n')

Running synthetic data...


[Alg1] [alpha=0.5] [lambda=0.1]: 100%|██████████| 96/96 [00:00<00:00, 22949.74it/s]


tensor([1.9661, 1.9713]) tensor([0.0506]) [1.9660925  1.9713432  0.05061606]


[ROAR] [alpha=0.5] [lambda=0.1]:  60%|██████    | 58/96 [00:32<00:21,  1.73it/s]

In [ ]:
df = d_results["synthetic"]
df[df["algorithm"]=="Alg1"]

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
0,Alg1,0,0.5,0.1,0,"[-2.3862, -1.7774]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
1,Alg1,0,0.5,0.1,1,"[-2.7105, -1.7402]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
2,Alg1,0,0.5,0.1,2,"[-2.3817, -1.9247]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
3,Alg1,0,0.5,0.1,3,"[-2.337, -1.0178]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
4,Alg1,0,0.5,0.1,4,"[-2.402, -2.7282]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
...,...,...,...,...,...,...,...,...
91,Alg1,0,0.5,0.1,91,"[-1.7741, -2.2538]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
92,Alg1,0,0.5,0.1,92,"[-2.2471, -3.4363]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
93,Alg1,0,0.5,0.1,93,"[-1.2901, -2.4369]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
94,Alg1,0,0.5,0.1,94,"[-2.911, -1.7206]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
